In [ ]:
import pickle
import re
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Loading data and features...")

Loading data and features...


In [ ]:
DATA_DIR = Path("dataset")
FEATURE_DIR = Path("features")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (75000, 4), Test: (75000, 3)


In [ ]:
def extract_pack_quantity(content: str) -> int:
    if not isinstance(content, str) or not content.strip():
        return 1

    patterns = [
        r"Pack of (\d+)",
        r"(\d+) per case",
        r"(\d+) count",
        r"(\d+)[- ]?pack",
        r"Set of (\d+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, content, flags=re.IGNORECASE)
        if match:
            return int(match.group(1))

    return 1


def extract_brand_prefix(content: str) -> str:
    if not isinstance(content, str):
        return "unknown"
    match = re.search(r"Item Name:\s*([^\n]+)", content, flags=re.IGNORECASE)
    if match:
        brand = match.group(1).strip().split(" ")[0]
        return brand.lower()
    return "unknown"


def compute_text_stats(df: pd.DataFrame) -> pd.DataFrame:
    stats = pd.DataFrame(index=df.index)
    text_series = df["catalog_content"].fillna("")

    stats["text_length"] = text_series.str.len()
    stats["text_word_count"] = text_series.str.split().str.len()
    stats["text_avg_word_length"] = (
        stats["text_length"] / stats["text_word_count"].replace(0, np.nan)
    ).fillna(0.0)
    stats["text_digit_count"] = text_series.str.count(r"\d")
    stats["text_upper_count"] = text_series.str.count(r"[A-Z]")
    stats["text_char_count_log"] = np.log1p(stats["text_length"])
    return stats


def clean_text_for_tfidf(text: str) -> str:
    if not isinstance(text, str):
        return ""
    lowered = text.lower()
    cleaned = re.sub(r"[^a-z0-9\s]", " ", lowered)
    return " ".join(cleaned.split())


# enrich train/test frames with EDA friendly columns
for frame in (train_df, test_df):
    frame["catalog_content"] = frame["catalog_content"].fillna("")
    frame["pack_quantity"] = frame.get("pack_quantity", np.nan)
    frame["brand_hint"] = frame["catalog_content"].apply(extract_brand_prefix)
    frame["derived_pack_quantity"] = frame["catalog_content"].apply(extract_pack_quantity)
    frame["pack_quantity"] = frame["pack_quantity"].fillna(frame["derived_pack_quantity"])
    frame.drop(columns=["derived_pack_quantity"], inplace=True)

train_text_stats = compute_text_stats(train_df)
test_text_stats = compute_text_stats(test_df)

train_df = pd.concat([train_df, train_text_stats], axis=1)
test_df = pd.concat([test_df, test_text_stats], axis=1)

train_df["log_price"] = np.log1p(train_df["price"].clip(lower=1e-6))

# Capture headline metrics for documentation
price_summary = train_df["price"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
missing_ratios = train_df.isna().mean().sort_values(ascending=False)
brand_top10 = train_df["brand_hint"].value_counts().head(10)

print(f"Price summary:\n{price_summary}")
print(f"Highest missing ratios:\n{missing_ratios.head(10)}")
print(f"Top 10 brand tokens:\n{brand_top10}")

Price summary:
count    75000.000000
mean        23.647654
std         33.376932
min          0.130000
10%          3.565000
25%          6.795000
50%         14.000000
75%         28.625000
90%         52.301000
95%         75.711000
max       2796.000000
Name: price, dtype: float64
Highest missing ratios:
sample_id               0.0
catalog_content         0.0
image_link              0.0
price                   0.0
pack_quantity           0.0
brand_hint              0.0
text_length             0.0
text_word_count         0.0
text_avg_word_length    0.0
text_digit_count        0.0
dtype: float64
Top 10 brand tokens:
brand_hint
food         985
the          636
mccormick    632
organic      438
rani         425
goya         424
frontier     366
la           341
betty        328
starbucks    303
Name: count, dtype: int64


In [ ]:
BASIC_FEATURE_COLUMNS = [
    "pack_quantity",
    "text_length",
    "text_word_count",
    "text_avg_word_length",
    "text_digit_count",
    "text_upper_count",
    "text_char_count_log"
]

missing_basic = [col for col in BASIC_FEATURE_COLUMNS if col not in train_df.columns]
if missing_basic:
    raise ValueError(f"Missing expected basic feature columns: {missing_basic}")

scaler_path = FEATURE_DIR / "feature_scaler.pkl"

if scaler_path.exists():
    print(f"Loading scaler from {scaler_path}")
    with scaler_path.open("rb") as handle:
        scaler = pickle.load(handle)
else:
    print("Fitting StandardScaler on metadata features")
    scaler = StandardScaler()
    scaler.fit(train_df[BASIC_FEATURE_COLUMNS])
    with scaler_path.open("wb") as handle:
        pickle.dump(scaler, handle)
        print(f"Scaler saved to {scaler_path}")

basic_features_train = scaler.transform(train_df[BASIC_FEATURE_COLUMNS])
basic_features_test = scaler.transform(test_df[BASIC_FEATURE_COLUMNS])

print(f"Basic features: train {basic_features_train.shape}, test {basic_features_test.shape}")

train_df["cleaned_text"] = train_df["catalog_content"].apply(clean_text_for_tfidf)
test_df["cleaned_text"] = test_df["catalog_content"].apply(clean_text_for_tfidf)

Loading scaler from features/feature_scaler.pkl
Basic features: train (75000, 7), test (75000, 7)


In [9]:
# Load text features
text_train = np.load(FEATURE_DIR / "text_features_enhanced_train.npy")
text_test = np.load(FEATURE_DIR / "text_features_enhanced_test.npy")
print(f"Text features: {text_train.shape}")

# Load image features
image_train = np.load(FEATURE_DIR / "image_features_train_full.npy")
image_test = np.load(FEATURE_DIR / "image_features_test_full.npy")
print(f"Image features: {image_train.shape}")

Text features: (75000, 806)
Image features: (75000, 512)


In [10]:
# Combine all features
X_train = sparse.hstack([
    sparse.csr_matrix(basic_features_train.astype(np.float32)),
    sparse.csr_matrix(text_train.astype(np.float32)),
    sparse.csr_matrix(image_train.astype(np.float32))
]).tocsr()

X_test = sparse.hstack([
    sparse.csr_matrix(basic_features_test.astype(np.float32)),
    sparse.csr_matrix(text_test.astype(np.float32)),
    sparse.csr_matrix(image_test.astype(np.float32))
]).tocsr()

y_train = train_df["price"].values.astype(np.float32)
y_train_log = np.log1p(y_train)

print(f"Final X_train: {X_train.shape}")
print(f"Final X_test: {X_test.shape}")

Final X_train: (75000, 1325)
Final X_test: (75000, 1325)


In [11]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.015,  # Low LR for better generalization
    'num_leaves': 127,  # leaves for complex patterns
    'max_depth': 12,
    'n_estimators': 8000,
    'subsample': 0.75,  # Less aggressive subsampling
    'colsample_bytree': 0.75,
    'min_child_samples': 15,  # Allow smaller leaf nodes
    'reg_alpha': 0.01,  # Less L1 regularization
    'reg_lambda': 0.1,  # Less L2 regularization
    'random_state': RANDOM_SEED,
    'verbose': -1,
    'device': 'gpu',
    'gpu_use_dp': False,
    'max_bin': 255
}

print("Training LightGBM....")
print(f"Learning rate: {lgb_params['learning_rate']}")
print(f"Iterations: {lgb_params['n_estimators']}")
print(f"Max depth: {lgb_params['max_depth']}")
print(f"Num leaves: {lgb_params['num_leaves']}")

Training LightGBM....
Learning rate: 0.015
Iterations: 8000
Max depth: 12
Num leaves: 127


In [12]:
model = lgb.LGBMRegressor(**lgb_params)
model.fit(X_train, y_train_log, eval_set=[(X_train, y_train_log)], callbacks=[lgb.log_evaluation(500)])

print("\nModel trained successfully!")


[500]	training's rmse: 0.535737
[1000]	training's rmse: 0.415469
[1500]	training's rmse: 0.337134
[2000]	training's rmse: 0.277126
[2500]	training's rmse: 0.230442
[3000]	training's rmse: 0.192837
[3500]	training's rmse: 0.16205
[4000]	training's rmse: 0.136717
[4500]	training's rmse: 0.115863
[5000]	training's rmse: 0.0984188
[5500]	training's rmse: 0.0838382
[6000]	training's rmse: 0.0716864
[6500]	training's rmse: 0.0614912
[7000]	training's rmse: 0.0528768
[7500]	training's rmse: 0.0454478
[8000]	training's rmse: 0.0392353

Model trained successfully!


In [13]:
# Save the trained model
model_path = FEATURE_DIR / "lgbm_model.pkl"

with model_path.open("wb") as handle:
    pickle.dump(model, handle)

print(f"Model saved to {model_path}")

Model saved to features/lgbm_model.pkl


In [ ]:
# Generate predictions
print("Generating predictions...")
test_log_preds = model.predict(X_test)
test_preds = np.maximum(np.expm1(test_log_preds), 0)
clip_lower, clip_upper = train_df["price"].quantile([0.001, 0.999]).values
test_preds = np.clip(test_preds, clip_lower, clip_upper)

print(f"Predictions: mean={test_preds.mean():.2f}, median={np.median(test_preds):.2f}")
print(f"Range: [{test_preds.min():.2f}, {test_preds.max():.2f}]")

Generating predictions...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Predictions: mean=18.63, median=13.72
Range: [0.71, 298.65]


In [15]:
# Save submission
submission = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "price": test_preds.astype(float)
})

output_path = Path("test_out.csv")
submission.to_csv(output_path, index=False)

print(f"\nSubmission saved to {output_path}")
print(f"Total rows: {len(submission)}")
print(f"Missing values: {submission.isnull().sum().sum()}")
print(f"Negative prices: {(submission['price'] < 0).sum()}")
print("\nFirst 10 predictions:")
print(submission.head(10))


Submission saved to test_out.csv
Total rows: 75000
Missing values: 0
Negative prices: 0

First 10 predictions:
   sample_id      price
0     100179  17.428498
1     245611  13.685390
2     146263  23.568079
3      95658   9.795205
4      36806  19.679831
5     148239   4.176944
6      92659   8.695097
7       3780   8.824964
8     196940  17.593959
9      20472   7.680504
